# Monte Carlo Methods

Define the problem (e.g., an integral, an optimization goal).

Generate random inputs.

Compute the function or process using the inputs.

Aggregate the results.

Increase sample size to refine accuracy.

In [1]:
from gymnasium import spaces
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random

## Potion Brewing Environment
#### The agent must move up to pick up a bottle, move down to put it in the brewing stand, move up to pick up an eye, move down to put it in the brewing stand, and retrieve the magic potion.

In [2]:
# Initializes the class, defining the environment.
class Potion_Bot:
    # Define action and observation space
    def __init__(this):

        # The current timestep
        this.time = 1

        # The state is a 4x1 grid,
        # Location of robot
        # The name of what's in the robot's hand (0 default),
        # What's in the brewing stand,
        # How many potions brewed
        this.state = [1, 0, 0, 0]

        # The action space is a magic number {0,1,2,3}
        # 0 is pick up
        # 1 is put down
        # 2 is move left
        # 3 is move right
        this.action_space = spaces.Discrete(4)

        # Total reward count
        # +10 to create a potion of magic
        # -3 to go to the garbage w/ nothing in hand
        # -1 on the wrong ingredient or in the wrong place
        # +1 to get the right ingredient or in the right place
        this.reward = 0

        this.terminal_state = 3


    # Executes one timestep within the environment
    # Input to the function is an action
    def step(this, a, print_please):

        # The state is a 4x1 grid,
        # Location of robot
        # The name of what's in the robot's hand (0 default),
        # What's in the brewing stand,
        # How many potions brewed

        # ["eye", "water", "brewing", "garbage"]

        ac = ""
            
        # Pick up
        if a == 0:
            ac = "PICK UP"
            # If the hand is already full
            if this.state[1] != 0:
                this.reward = this.reward - 1
            # If we pick up an eye successfully
            elif this.state[0] == 0:
                # Pick up the eye
                this.state[1] = "eye"
                # If the brewing stand is ready for it, give rewards
                if this.state[2] == "bottle":
                    this.reward = this.reward + 1
                else:
                    this.reward = this.reward - 1
            # If we pick up water
            elif this.state[0] == 1:
                # Pick up the bottle
                this.state[1] = "bottle"
                # If the brewing stand is ready for it, give rewards
                if this.state[2] == 0:
                    this.reward = this.reward + 1
                else:
                    this.reward = this.reward - 1
            # If we pick up whatever's in the brewing stand
            elif this.state[0] == 2:
                # If the potion is completed
                if this.state[2] == "magic":
                    this.state[2] = 0
                    this.state[3] = this.state[3] + 1
                    this.reward = this.reward + 10
                # If the agent tries to pick up the bottle
                elif this.state[2] == "bottle":
                    this.state[1] = "bottle"
                    this.state[2] = 0
                    this.reward = this.reward - 1
                # Otherwise it tries to pick up an empty brewing stand which is bad. give punishment
                else:
                    this.reward = this.reward - 1
            else:
                this.reward = this.reward - 1

        # Put down
        elif a == 1:
            ac = "PUT DOWN"
            # If there's nothing in his hand, there's nothing to put down
            if this.state[1] == 0:
                this.reward = this.reward - 1
            # If he's throwing something out
            elif this.state[0] == 3:
                # Just throw it out and don't adjust points
                this.state[1] = 0
            # Nothing can be put down on the eye stand or water stand, only picked up
            elif this.state[0] == 0 or this.state[0] == 1:
                this.reward = this.reward - 1
            # he is at the brewing stand & there is something in his hand
            else:
                # If he's putting a bottle into an empty brewing stand
                if this.state[1] == "bottle" and this.state[2] == 0:
                    this.state[1] = 0
                    this.state[2] = "bottle"
                    this.reward = this.reward + 1
                # If he's adding an eye to a brewing stand with just a bottle
                elif this.state[1] == "eye" and this.state[2] == "bottle":
                    this.state[1] = 0
                    this.state[2] = "magic"
                    this.reward = this.reward + 1
                # Something has gone wrong so do nothing
                else:
                    this.reward = this.reward - 1

        # Move left
        elif a == 2:
            ac = "MOVE UP"
            # He can't move left anymore
            if this.state[0] == 0:
                this.reward = this.reward - 1
            # Move him left
            else:
                this.state[0] = this.state[0] - 1
            

        # The state is a 4x1 grid,
        # Location of robot
        # The name of what's in the robot's hand (0 default),
        # What's in the brewing stand,
        # How many potions brewed

        # ["eye", "water", "brewing", "garbage"]
            
        # Move right
        elif a == 3:
            ac = "MOVE DOWN"
            # He can't move left anymore
            if this.state[0] == 3:
                this.reward = this.reward - 1
            # Move him left
            else:
                this.state[0] = this.state[0] + 1

        if print_please:
            print(str(this.time)+": Action: "+ac)
            print(str(this.time)+": REWARD: "+str(this.reward))
            this.render()
        # If enough people were dropped off such that the terminal state is complete, return true
        if this.state[3] == this.terminal_state:
            return True
            
        this.time = this.time + 1

        return False


    # Resets the state of the environment to an initial state
    def reset(this):
        new_environment = Potion_Bot()
        this.state = new_environment.state
        this.time = 1
        this.rewards = 0
        this.terminal_state = 3

    # Visualizes the environment
    # Any form like vector representation or visualizing using matplotlib will be sufficient
    def render(this):
        print("")
        print("Laboratory: ")
        print("________________")
        for i in range(4):
            subtext = ""
            if i == 0:
                subtext = " eyes"
            elif i == 1:
                subtext = " bottles"
            elif i == 2:
                if this.state[2] != 0:
                    subtext = " brewing stand "+this.state[2]
                else:
                    subtext = " brewing stand: EMPTY"
            else:
                subtext = " trash can"

            if i != this.state[0]:
                print("|        |"+subtext)
            elif this.state[1] == "bottle":
                print("| bottle |"+subtext)
            elif this.state[1] == "eye":
                print("|   eye  |"+subtext)
            else:
                print("|   []   |"+subtext)
        print("________________")
        print("Potions Collected: "+str(this.state[3]))
        print("")
        print("")

## Action Simulator
#### Steps are considered 'real' because the states are printed to the console.

In [3]:
# This sends in a set of steps, which is an array of integers [0,3] representing moves to be executed with print statements
def environment_simulate(steps):
    env = Potion_Bot()
    env.render()

                # 0 is idle
                # 1 is pick up
                # 2 is put down
                # 3 is move up
                # 4 is move down

    for step in steps:
        env.step(step, True)

## Epsilon Greedy algorithm

#### Maintains a Q-table and trains the model

In [4]:
def epsilon_greedy(epsilon, episodes, discount, learning_rate, print_please):

    env = Potion_Bot()
    Q = {} # map (s,a) to Q-value => int
    for o in range(episodes):
        terminal = False
        while not terminal:
            s = tuple(env.state)
            pre_reward = env.reward
            # Dynamically add the states to Q
            if (s,0) not in Q:
                Q[(s,0)] = 0
                Q[(s,1)] = 0
                Q[(s,2)] = 0
                Q[(s,3)] = 0
            
            # Initialize action
            action = -1

            # Explore
            if random.random() <= epsilon:
                action = random.randint(0,3)
            # Exploit
            else:
                max_action = -math.inf
                for i in range(4):
                    if Q[(s,i)] > max_action:
                        max_action = Q[(s,i)]
                        action = i

            # Take the step and see if it's terminal
            if env.step(action, False):
                # Terminal State
                terminal = True

            # Observe s' and R
            R = env.reward - pre_reward
            s_prime = tuple(env.state)

            # If s_prime hasn't been seen yet, add the default
            if (s_prime,0) not in Q:
                Q[(s_prime,0)] = 0
                Q[(s_prime,1)] = 0
                Q[(s_prime,2)] = 0
                Q[(s_prime,3)] = 0

            # Find the BEST action for s'
            best_action_for_s_prime = -1
            max_action = -math.inf
            for i in range(4):
                if Q[(s_prime,i)] > max_action:
                    max_action = Q[(s_prime,i)]
                    best_action_for_s_prime = i

            # Update the Q-table
            Q[(s, action)] = Q[(s, action)] + learning_rate*(R + discount*Q[(s_prime, best_action_for_s_prime)] - Q[(s, action)])  

        # Print the Q-table update if requested in the parameters.
        if print_please:
            for key in Q.keys():
                print("Episode: "+str(o+1))
                print(str(key)+": "+str(Q[key]))
                print("")

        # Reset the env
        env.reset()  

    return Q


## Epsilon Greedy Training & Testing

#### Run this to see the model (Q) be trained and executed until three potions are made

In [5]:
# Train the model (Q)
Q = epsilon_greedy(epsilon=0.2, episodes=10, discount=1, learning_rate=.5, print_please=False)

# Test it on a new environment
env = Potion_Bot()

# Break the loop when it reaches a terminal state
terminal = False
while not terminal:
    # the key always must be a tuple (no lists can be included in the key of a dict)
    s = tuple(env.state)
    if (s,0) not in Q:
        Q[(s,0)] = 0
        Q[(s,1)] = 0
        Q[(s,2)] = 0
        Q[(s,3)] = 0

    a = -1
    max_action = -math.inf
    for i in range(4):
        if Q[(s,i)] > max_action:
            max_action = Q[(s,i)]
            # Set a to the argmax_a(Q[(s,a)])
            a = i
            
    # Make the step & say if it's terminal
    terminal = env.step(a, print_please=True)

    


1: Action: PICK UP
1: REWARD: 1

Laboratory: 
________________
|        | eyes
| bottle | bottles
|        | brewing stand: EMPTY
|        | trash can
________________
Potions Collected: 0


2: Action: MOVE DOWN
2: REWARD: 1

Laboratory: 
________________
|        | eyes
|        | bottles
| bottle | brewing stand: EMPTY
|        | trash can
________________
Potions Collected: 0


3: Action: PUT DOWN
3: REWARD: 2

Laboratory: 
________________
|        | eyes
|        | bottles
|   []   | brewing stand bottle
|        | trash can
________________
Potions Collected: 0


4: Action: MOVE UP
4: REWARD: 2

Laboratory: 
________________
|        | eyes
|   []   | bottles
|        | brewing stand bottle
|        | trash can
________________
Potions Collected: 0


5: Action: MOVE UP
5: REWARD: 2

Laboratory: 
________________
|   []   | eyes
|        | bottles
|        | brewing stand bottle
|        | trash can
________________
Potions Collected: 0


6: Action: PICK UP
6: REWARD: 3

Laborator

## Attempt on Monte Carlo

#### ADI: I couldn't get much further. I'm using the notes from Alena. Let's complete this more Thursday.

In [7]:
# Try. I'm not sure how far this will take me or if I'm even on the right track.
def monte_carlo(epsilon, episodes, discount, learning_rate):
    # Train the model
    Q = epsilon_greedy(epsilon=epsilon, episodes=10, discount=discount, learning_rate=learning_rate, print_please=False)

    # Keep a list of all observed states (but more can come later so S won't be extensive!)
    S = set()
    for key in Q.keys():
        if key[0] not in S:
            S.add(key[0])

    # Value function mapping S to expected reward
    V = {}

    env = Potion_Bot()

    for i in range(episodes):
        # Randomly choose a state from s -- the state should NOT be terminal. If it is, don't evaluate it
        s = S.pop()
        if s not in V:
            V[s] = 0

        # Initialze the model
        env.state = list(s)
        # If s is not terminal
        if s[3] != env.terminal_state:
            cumul_reward = 0
            terminal = False
            while not terminal:

                a = -1
                max_action = -math.inf
                for i in range(4):
                    if Q[(s,i)] > max_action:
                        max_action = Q[(s,i)]
                        # Set a to the argmax_a(Q[(s,a)])
                        a = i

                pre_reward = env.reward

                # Take the step and check to make sure it is not terminal
                terminal = env.step(a, False)

                # Add the cumulative reward
                cumul_reward = cumul_reward + (env.reward - pre_reward)

            # Update the value function given the start state and the cumulative reward
            V[s] = V[s] + learning_rate*(cumul_reward - V[s])

            # Reset the model
            env.reset()
    return V

# Idk
print(monte_carlo(epsilon=0.2, episodes=10, discount=1, learning_rate=.5))

KeyboardInterrupt: 

## My previous version of monte carlo that will probably be deleted.

In [ ]:
# Implements a simple monte carlo
# Has a million episodes with each episode running ten moves, and returning the action series with the highest reward
def delete_me_monte_carlo(episodes):
    
    actions_to_rewards = {}
    
    # For every episode
    for _ in range(episodes):
        # Initialize a new environment (you can just reset this later for time purposes)
        env = Potion_Bot()
        # Generate ten random moves
        actions = [random.randint(0,3) for _ in range(10)]
        # Execute each action
        for action in actions:
            #pre = env.reward
            env.step(action, False)
            #post = env.reward

        # Append the steps to the reward key
        if env.reward in actions_to_rewards.keys():
            cur = actions_to_rewards[env.reward] 
            cur.append(actions)
            actions_to_rewards[env.reward] = cur
        else:
            actions_to_rewards[env.reward] = [actions]
 
    # keep track of what the highest reward is
    highest_key = -math.inf
    for key in actions_to_rewards.keys():
        if key > highest_key:
            highest_key = key
    
    # Return the action steps that led to the highest reward
    return actions_to_rewards[highest_key]